In [ ]:
!pip install pymatgen mp-api torch-geometric

import argparse
import random
from dataclasses import dataclass, field
from typing import Optional, List, Tuple
import numpy as np
import torch
import torch.nn as nn

from pymatgen.core import Structure
from mp_api.client import MPRester

from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import NNConv

# config
MP_API_KEY = "xxxxx"
SEED = 318

NUM_STRUCTURES = 50
SHOCKS_PER_STRUCTURE = 60
CUTOFF = 4.0
NEIGHBOR_CUTOFF = 4.5
INTERACTION_CUTOFF = 4.0
RBF_DIM = 8
SHOCK_STRENGTHS = np.linspace(0.1, 1.5, 10)
SHOCK_DIRECTIONS = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1],
], dtype=np.float32)

DISPLACEMENT_SCALE_A = 0.50

BATCH_SIZE = 16
EPOCHS = 100
LR = 0.001

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


@dataclass
class EdgeSpec:
    """Explicit metadata specification for graph edge feature layout."""
    vector_slice: Tuple[int, int] = (0, 3)
    angle_index: Optional[int] = None
    scalar_features: List[int] = field(default_factory=list)


def cosine_cutoff(distances, cutoff):
    """
    Smooth cosine envelope:
        f(r) = 0.5 * [cos(pi*r/cutoff) + 1]
        for r < cutoff
        0 otherwise

    f(r) -> 0 continuously as r -> cutoff.
    """
    x = distances / cutoff

    return torch.where(
        x < 1.0,
        0.5 * (torch.cos(torch.pi * x) + 1.0),
        torch.zeros_like(x)
    )


def radial_basis(distances, cutoff, num_basis=8):
    """
    Gaussian radial basis expansion.
    """
    centers = torch.linspace(
        0.0,
        cutoff,
        num_basis,
        device=distances.device,
        dtype=distances.dtype
    )

    width = cutoff / (num_basis - 1)

    return torch.exp(
        -0.5 * (
            (distances.unsqueeze(1) - centers.unsqueeze(0))
            / width
        ) ** 2
    )


# geometric math helpers
def random_rotation_matrix():
    q, r = torch.linalg.qr(torch.randn(3, 3))
    d = torch.diag(torch.sign(torch.diag(r)))
    q = q @ d
    if torch.det(q) < 0:
        q[:, 0] = -q[:, 0]
    return q


def apply_rotation(data, R):
    data_rot = data.clone()

    R = R.to(data.pos.device)

    # 1. rotate positions
    data_rot.pos = data.pos @ R

    # 2. rotate shock direction vector
    dir_orig = data.x[0, 2:5]
    dir_rot = dir_orig @ R
    s_hat = dir_rot / (torch.norm(dir_rot) + 1e-6)

    # 3. recompute node features (specifically direction and depth)
    x_rot = data.x.clone()
    x_rot[:, 2:5] = dir_rot.repeat(data.num_nodes, 1)

    # recompute normalized depth from rotated positions and rotated direction
    depth = data_rot.pos @ s_hat
    depth = depth - depth.min()
    depth_norm = depth / (depth.max() - depth.min() + 1e-6)
    x_rot[:, 5] = depth_norm

    data_rot.x = x_rot

    # Extract explicit feature layout spec
    edge_spec_raw = getattr(data, "edge_spec", EdgeSpec())
    edge_spec = edge_spec_raw[0] if isinstance(edge_spec_raw, list) else edge_spec_raw
    v_start, v_end = edge_spec.vector_slice
    angle_idx = edge_spec.angle_index

    # 4. rotate edge attribute vectors (only vector slice)
    edge_attr_rot = data.edge_attr.clone()
    edge_vectors = data.edge_attr[:, v_start:v_end] @ R
    edge_attr_rot[:, v_start:v_end] = edge_vectors

    # 5. recompute cos(theta) = r_hat . s_hat only if explicitly present
    if angle_idx is not None:
        cos_theta = (edge_vectors * s_hat.unsqueeze(0)).sum(dim=-1, keepdim=True)
        edge_attr_rot[:, angle_idx:angle_idx + 1] = cos_theta

    data_rot.edge_attr = edge_attr_rot

    # 6. rotate targets
    data_rot.y = data.y @ R
    return data_rot


def apply_rotation_to_batch(batch):
    """
    Applies independent random 3D rotations to each graph in a PyG Batch
    without unbatching or reconstructing DataLoaders.
    """
    batch_rot = batch.clone()
    num_graphs = batch.num_graphs

    # Generate random rotations for each graph in the batch
    rotations = torch.stack([random_rotation_matrix() for _ in range(num_graphs)]).to(batch.pos.device)

    # Pre-extract per-node rotation matrices and per-edge rotation matrices
    node_R = rotations[batch.batch]  # [N, 3, 3]
    edge_R = rotations[batch.batch[batch.edge_index[0]]]  # [E, 3, 3]

    # 1. Rotate positions: pos_i = pos_i @ R_b
    batch_rot.pos = torch.bmm(batch.pos.unsqueeze(1), node_R).squeeze(1)

    # 2. Rotate shock directions and update node features
    dir_orig = batch.x[:, 2:5]
    dir_rot = torch.bmm(dir_orig.unsqueeze(1), node_R).squeeze(1)

    x_rot = batch.x.clone()
    x_rot[:, 2:5] = dir_rot

    # Recompute depth per graph
    s_hat = dir_rot / (torch.norm(dir_rot, dim=1, keepdim=True) + 1e-6)
    depth = (batch_rot.pos * s_hat).sum(dim=1)

    # Normalize depth per graph in the batch
    for g_idx in range(num_graphs):
        mask = (batch.batch == g_idx)
        g_depth = depth[mask]
        g_min, g_max = g_depth.min(), g_depth.max()
        depth_norm = (g_depth - g_min) / (g_max - g_min + 1e-6)
        x_rot[mask, 5] = depth_norm

    batch_rot.x = x_rot

    # FIX 1: Safely unwrap EdgeSpec list attribute generated by PyG Batch collation
    edge_spec_raw = getattr(batch, "edge_spec", EdgeSpec())
    edge_spec = edge_spec_raw[0] if isinstance(edge_spec_raw, list) and len(edge_spec_raw) > 0 else (edge_spec_raw if not isinstance(edge_spec_raw, list) else EdgeSpec())

    v_start, v_end = edge_spec.vector_slice
    angle_idx = edge_spec.angle_index

    # 3. Rotate edge vectors and update edge attributes
    edge_vectors_orig = batch.edge_attr[:, v_start:v_end]
    edge_vectors_rot = torch.bmm(edge_vectors_orig.unsqueeze(1), edge_R).squeeze(1)

    edge_attr_rot = batch.edge_attr.clone()
    edge_attr_rot[:, v_start:v_end] = edge_vectors_rot

    # 4. If angular feature present, recompute cos(theta) = r_hat . s_hat
    if angle_idx is not None:
        edge_s_hat = s_hat[batch.batch[batch.edge_index[0]]]
        cos_theta = (edge_vectors_rot * edge_s_hat).sum(dim=-1, keepdim=True)
        edge_attr_rot[:, angle_idx:angle_idx + 1] = cos_theta

    batch_rot.edge_attr = edge_attr_rot

    # 5. Rotate target vectors
    batch_rot.y = torch.bmm(batch.y.unsqueeze(1), node_R).squeeze(1)

    return batch_rot


def add_angular_features(data):
    data = data.clone()
    r_hat = data.edge_attr[:, :3]
    s = data.x[0, 2:5]
    s_hat = s / (torch.norm(s) + 1e-6)

    cos_theta = (r_hat * s_hat.unsqueeze(0)).sum(dim=-1, keepdim=True)
    angle_idx = data.edge_attr.shape[1]
    data.edge_attr = torch.cat([data.edge_attr, cos_theta], dim=1)

    old_spec_raw = getattr(data, "edge_spec", EdgeSpec())
    old_spec = old_spec_raw[0] if isinstance(old_spec_raw, list) else old_spec_raw
    data.edge_spec = EdgeSpec(
        vector_slice=old_spec.vector_slice,
        angle_index=angle_idx,
        scalar_features=old_spec.scalar_features
    )
    return data


def rotation_sanity_check(sample_graph):
    print("\nStarting rotation sanity checks across edge representations...")
    modes = ["baseline", "angle", "smooth_angle", "rbf_angle", "smooth_rbf_angle"]

    for mode in modes:
        test_graph = sample_graph.clone()
        if mode == "angle":
            test_graph = add_angular_features(test_graph)

        R = random_rotation_matrix()
        rotated = apply_rotation(test_graph, R)

        ortho_err = torch.max(torch.abs(R.T @ R - torch.eye(3))).item()
        det_val = torch.det(R).item()

        spec = test_graph.edge_spec[0] if isinstance(test_graph.edge_spec, list) else test_graph.edge_spec
        v_start, v_end = spec.vector_slice
        edge_vec_orig = test_graph.edge_attr[:, v_start:v_end]
        edge_vec_rot = rotated.edge_attr[:, v_start:v_end]
        edge_vector_err = torch.max(torch.abs(edge_vec_rot - edge_vec_orig @ R)).item()

        edge_norm_orig = torch.norm(edge_vec_orig, dim=1)
        edge_norm_rot = torch.norm(edge_vec_rot, dim=1)
        max_edge_norm_err = torch.max(torch.abs(edge_norm_orig - edge_norm_rot)).item()

        dir_orig = test_graph.x[0, 2:5]
        dir_rot = rotated.x[0, 2:5]
        shock_norm_err = torch.abs(torch.norm(dir_orig) - torch.norm(dir_rot)).item()

        target_norm_orig = torch.norm(test_graph.y, dim=1)
        target_norm_rot = torch.norm(rotated.y, dim=1)
        target_norm_err = torch.max(torch.abs(target_norm_orig - target_norm_rot)).item()

        depth_orig = test_graph.x[:, 5]
        depth_rot = rotated.x[:, 5]
        depth_err = torch.max(torch.abs(depth_orig - depth_rot)).item()

        scalars_orig = test_graph.edge_attr[:, spec.scalar_features]
        scalars_rot = rotated.edge_attr[:, spec.scalar_features]
        scalar_err = torch.max(torch.abs(scalars_orig - scalars_rot)).item() if len(spec.scalar_features) > 0 else 0.0

        angle_err = 0.0
        if spec.angle_index is not None:
            a_idx = spec.angle_index
            angle_err = torch.max(torch.abs(test_graph.edge_attr[:, a_idx] - rotated.edge_attr[:, a_idx])).item()

        print(f"\nMode: {mode}")
        print(f"  orthogonality error:  {ortho_err:.2e}")
        print(f"  det(R):               {det_val:.6f}")
        print(f"  edge vector transform: {edge_vector_err:.2e}")
        print(f"  edge norm error:       {max_edge_norm_err:.2e}")
        print(f"  shock norm error:      {shock_norm_err:.2e}")
        print(f"  target norm error:     {target_norm_err:.2e}")
        print(f"  recomputed depth err:  {depth_err:.2e}")
        print(f"  scalar feature err:    {scalar_err:.2e}")
        print(f"  angle invariance err:   {angle_err:.2e}")


# synthetic shock model
def synthetic_shock_response(positions, atomic_numbers, edge_index, neighbor_vectors, neighbor_distances, coordination, strength, direction):
    direction = direction / torch.norm(direction)
    depth = positions @ direction
    depth = depth - depth.min()
    depth_norm = depth / (depth.max() - depth.min() + 1e-6)

    attenuation = torch.exp(-1.2 * depth_norm)
    stiffness = (0.65 + 0.35 * coordination)
    nonlinear_strength = (strength + 0.35 * strength ** 2)

    longitudinal = (nonlinear_strength * attenuation / stiffness)
    longitudinal = (longitudinal * DISPLACEMENT_SCALE_A).unsqueeze(1)

    parallel_component = (neighbor_vectors @ direction).unsqueeze(1) * direction
    transverse_vectors = (neighbor_vectors - parallel_component)
    transverse_norm = (torch.norm(transverse_vectors, dim=1, keepdim=True) + 1e-6)
    transverse_vectors = (transverse_vectors / transverse_norm)

    atom_indices = edge_index[0]
    local_transverse = torch.zeros((positions.shape[0], 3), dtype=positions.dtype)
    local_transverse.index_add_(0, atom_indices, transverse_vectors)

    local_counts = torch.zeros((positions.shape[0], 1), dtype=positions.dtype)
    local_counts.index_add_(0, atom_indices, torch.ones((len(atom_indices), 1), dtype=positions.dtype))

    local_transverse = (local_transverse / (local_counts + 1e-6))
    local_norm = (torch.norm(local_transverse, dim=1, keepdim=True) + 1e-6)
    local_transverse = (local_transverse / local_norm)

    lateral = (-0.18 * longitudinal * local_transverse)
    local_variation = (0.04 * strength * (coordination - 1.0) * DISPLACEMENT_SCALE_A).unsqueeze(1)

    displacement = (longitudinal * direction + lateral + local_variation * direction)
    return displacement


# materials project
def get_structures():
    with MPRester(MP_API_KEY) as mpr:
        docs = mpr.materials.summary.search(
            formula="SiO2",
            fields=["material_id", "structure"],
            chunk_size=50
        )
    structures = []
    for doc in docs[:NUM_STRUCTURES]:
        structures.append({
            "material_id": str(doc.material_id),
            "structure": doc.structure
        })
    print(f"structures retrieved: {len(structures)}")
    return structures


# graph construction
def structure_to_graph(
    structure,
    material_index,
    material_id,
    strength,
    direction,
    direction_id, # FIX 3: passed direct index logic to avert empty floating point arrays
    edge_mode="baseline"
):
    coords = torch.tensor(
        structure.cart_coords,
        dtype=torch.float32
    )

    atomic_numbers = torch.tensor(
        [site.specie.Z for site in structure.sites],
        dtype=torch.float32
    )

    # Build graph using NEIGHBOR_CUTOFF
    center_indices, neighbor_indices, images, distances = (
        structure.get_neighbor_list(NEIGHBOR_CUTOFF)
    )

    center_indices = torch.tensor(center_indices, dtype=torch.long)
    neighbor_indices = torch.tensor(neighbor_indices, dtype=torch.long)
    distances = torch.tensor(distances, dtype=torch.float32)
    images = torch.tensor(images, dtype=torch.float32)
    lattice = torch.tensor(structure.lattice.matrix, dtype=torch.float32)
    frac_coords = torch.tensor(structure.frac_coords, dtype=torch.float32)

    neighbor_coords = frac_coords[neighbor_indices] + images @ lattice
    neighbor_vectors = neighbor_coords - coords[center_indices]
    normalized_vectors = neighbor_vectors / (distances.unsqueeze(1) + 1e-6)

    # ---------------------------------------------------------
    # Edge representation
    # ---------------------------------------------------------

    if edge_mode == "baseline":
        radial = (distances / INTERACTION_CUTOFF).unsqueeze(1)
        edge_attr = torch.cat([normalized_vectors, radial], dim=1)
        edge_spec = EdgeSpec(vector_slice=(0, 3), angle_index=None, scalar_features=[3])

    elif edge_mode == "angle":
        radial = (distances / INTERACTION_CUTOFF).unsqueeze(1)
        d = torch.tensor(direction, dtype=torch.float32)
        d = d / torch.norm(d)
        cos_theta = (normalized_vectors * d.unsqueeze(0)).sum(dim=1, keepdim=True)

        edge_attr = torch.cat([normalized_vectors, radial, cos_theta], dim=1)
        edge_spec = EdgeSpec(vector_slice=(0, 3), angle_index=4, scalar_features=[3])

    elif edge_mode == "smooth_angle":
        envelope = cosine_cutoff(distances, INTERACTION_CUTOFF).unsqueeze(1)
        d = torch.tensor(direction, dtype=torch.float32)
        d = d / torch.norm(d)
        cos_theta = (normalized_vectors * d.unsqueeze(0)).sum(dim=1, keepdim=True)

        edge_attr = torch.cat([
            normalized_vectors,
            (distances / INTERACTION_CUTOFF).unsqueeze(1),
            cos_theta,
            envelope
        ], dim=1)
        edge_spec = EdgeSpec(vector_slice=(0, 3), angle_index=4, scalar_features=[3, 5])

    elif edge_mode == "rbf_angle":
        rbf = radial_basis(distances, INTERACTION_CUTOFF, RBF_DIM)
        d = torch.tensor(direction, dtype=torch.float32)
        d = d / torch.norm(d)
        cos_theta = (normalized_vectors * d.unsqueeze(0)).sum(dim=1, keepdim=True)

        edge_attr = torch.cat([normalized_vectors, rbf, cos_theta], dim=1)
        edge_spec = EdgeSpec(
            vector_slice=(0, 3),
            angle_index=3 + RBF_DIM,
            scalar_features=list(range(3, 3 + RBF_DIM))
        )

    elif edge_mode == "smooth_rbf_angle":
        envelope = cosine_cutoff(distances, INTERACTION_CUTOFF).unsqueeze(1)
        rbf = radial_basis(distances, INTERACTION_CUTOFF, RBF_DIM)
        d = torch.tensor(direction, dtype=torch.float32)
        d = d / torch.norm(d)
        cos_theta = (normalized_vectors * d.unsqueeze(0)).sum(dim=1, keepdim=True)

        edge_attr = torch.cat([
            normalized_vectors,
            rbf * envelope,
            cos_theta,
            envelope
        ], dim=1)
        edge_spec = EdgeSpec(
            vector_slice=(0, 3),
            angle_index=3 + RBF_DIM,
            scalar_features=list(range(3, 3 + RBF_DIM)) + [3 + RBF_DIM + 1]
        )

    else:
        raise ValueError(f"Unknown edge_mode: {edge_mode}")

    edge_index = torch.stack([center_indices, neighbor_indices], dim=0)

    # ---------------------------------------------------------
    # Coordination
    # ---------------------------------------------------------

    coordination = torch.zeros(len(structure), dtype=torch.float32)
    coordination.index_add_(0, center_indices, torch.ones(len(center_indices), dtype=torch.float32))
    coordination = torch.clamp(coordination / 10.0, 0.0, 1.0)

    # ---------------------------------------------------------
    # Shock direction / depth
    # ---------------------------------------------------------

    d = torch.tensor(direction, dtype=torch.float32)
    d = d / torch.norm(d)

    depth = coords @ d
    depth = depth - depth.min()
    depth = depth / (depth.max() - depth.min() + 1e-6)

    atomic_norm = atomic_numbers / 100.0
    strength_feature = torch.full((len(structure), 1), float(strength))
    direction_feature = d.unsqueeze(0).repeat(len(structure), 1)

    node_features = torch.cat([
        atomic_norm.unsqueeze(1),
        strength_feature,
        direction_feature,
        depth.unsqueeze(1),
        coordination.unsqueeze(1)
    ], dim=1)

    # ---------------------------------------------------------
    # Synthetic target
    # ---------------------------------------------------------

    displacement = synthetic_shock_response(
        positions=coords,
        atomic_numbers=atomic_numbers,
        edge_index=edge_index,
        neighbor_vectors=neighbor_vectors,
        neighbor_distances=distances,
        coordination=coordination,
        strength=torch.tensor(float(strength), dtype=torch.float32),
        direction=d
    )

    data = Data(
        x=node_features,
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=displacement,
        pos=coords
    )

    data.edge_spec = edge_spec
    data.material_index = material_index
    data.material_id = material_id
    data.shock_strength = float(strength)
    data.direction_id = direction_id
    data.direction = direction.tolist()
    return data


def build_dataset(structures, edge_mode="baseline"):
    graphs = []
    conditions = []

    # Passing direct enumerations to safely index conditions without float errors later
    for dir_idx, direction in enumerate(SHOCK_DIRECTIONS):
        for strength in SHOCK_STRENGTHS:
            conditions.append((strength, direction, dir_idx))

    conditions = conditions[:SHOCKS_PER_STRUCTURE]

    for material_index, item in enumerate(structures):
        print(f"building material {material_index + 1}/{len(structures)}")
        for strength, direction, dir_idx in conditions:
            graph = structure_to_graph(
                structure=item["structure"],
                material_index=material_index,
                material_id=item["material_id"],
                strength=strength,
                direction=direction,
                direction_id=dir_idx,
                edge_mode=edge_mode
            )
            graphs.append(graph)

    print(f"\ngraphs: {len(graphs)}")
    return graphs


# model
class ShockGNN(nn.Module):
    def __init__(self, node_dim=7, edge_dim=4, hidden_dim=64):
        super().__init__()

        edge_network_1 = nn.Sequential(
            nn.Linear(edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim * hidden_dim)
        )

        self.conv1 = NNConv(node_dim, hidden_dim, edge_network_1, aggr="mean")

        edge_network_2 = nn.Sequential(
            nn.Linear(edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * hidden_dim)
        )

        self.conv2 = NNConv(hidden_dim, hidden_dim, edge_network_2, aggr="mean")

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 3)
        )

    def forward(self, x, edge_index, edge_attr):
        x = self.conv1(x, edge_index, edge_attr)
        x = torch.relu(x)
        x = self.conv2(x, edge_index, edge_attr)
        x = torch.relu(x)
        return self.head(x)


# train
def train_model(model, train_loader, test_loader, use_augmentation=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_nodes = 0

        for batch in train_loader:
            if use_augmentation:
                # Optimized tensor-native batch rotation
                batch = apply_rotation_to_batch(batch)

            batch = batch.to(device)
            optimizer.zero_grad()

            prediction = model(batch.x, batch.edge_index, batch.edge_attr)
            loss = loss_fn(prediction, batch.y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch.y.numel()
            total_nodes += batch.y.numel()

        train_loss = total_loss / total_nodes
        test_loss = evaluate_mse(model, test_loader)

        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | train={train_loss:.6f} | test={test_loss:.6f}")

    return model


# metrics
@torch.no_grad()
def evaluate_mse(model, loader):
    model.eval()
    total = 0.0
    count = 0

    for batch in loader:
        batch = batch.to(device)
        prediction = model(batch.x, batch.edge_index, batch.edge_attr)
        error = prediction - batch.y
        total += torch.sum(error ** 2).item()
        count += batch.y.numel()

    return total / count


@torch.no_grad()
def evaluate_metrics(model, loader):
    model.eval()
    absolute_error = 0.0
    squared_error = 0.0
    count = 0

    predicted_magnitude = []
    actual_magnitude = []

    for batch in loader:
        batch = batch.to(device)
        prediction = model(batch.x, batch.edge_index, batch.edge_attr)
        error = prediction - batch.y

        absolute_error += torch.sum(torch.abs(error)).item()
        squared_error += torch.sum(error ** 2).item()
        count += batch.y.numel()

        predicted_magnitude.append(torch.norm(prediction, dim=1).cpu())
        actual_magnitude.append(torch.norm(batch.y, dim=1).cpu())

    mae = absolute_error / count
    rmse = np.sqrt(squared_error / count)

    predicted_magnitude = torch.cat(predicted_magnitude)
    actual_magnitude = torch.cat(actual_magnitude)

    return {
        "mae": mae,
        "rmse": rmse,
        "mean_pred": predicted_magnitude.mean().item(),
        "mean_actual": actual_magnitude.mean().item(),
        "max_pred": predicted_magnitude.max().item(),
        "max_actual": actual_magnitude.max().item()
    }


@torch.no_grad()
def compute_rotation_consistency_error(model, loader):
    """
    Computes the empirical rotation-consistency error ||f(RG) - f(G)R|| / ||f(G)||,
    evaluating how well the model outputs transform predictably when the inputs rotate.
    """
    model.eval()
    total_diff = 0.0
    total_norm = 0.0

    for batch in loader:
        data_list = batch.to_data_list()
        for g in data_list:
            g_dev = g.to(device)

            f_G = model(
                g_dev.x,
                g_dev.edge_index,
                g_dev.edge_attr
            )

            R = random_rotation_matrix().to(device)

            g_rot = apply_rotation(g_dev, R)

            f_RG = model(
                g_rot.x,
                g_rot.edge_index,
                g_rot.edge_attr
            )

            Rf_G = f_G @ R
            diff = torch.norm(f_RG - Rf_G, dim=1).sum().item()
            norm = torch.norm(f_G, dim=1).sum().item()

            total_diff += diff
            total_norm += norm

    return total_diff / (total_norm + 1e-6)


# splits
def direction_split(graphs):
    directions = sorted(set(g.direction_id for g in graphs))
    test_directions = set(directions[::2])

    train, test = [], []
    for i, g in enumerate(graphs):
        if g.direction_id in test_directions:
            test.append(i)
        else:
            train.append(i)

    return np.array(train), np.array(test)


# experiment runner
def run_b_experiment(name, graphs, train_indices, test_indices, use_augmentation=False, use_angular=False):
    print("\n" + "=" * 70)
    print(f"{name}")
    print(f"Rotation augmentation: {'ON' if use_augmentation else 'OFF'}")
    print(f"Angular features:      {'ON' if use_angular else 'OFF'}")
    print("=" * 70)

    print(f"train graphs: {len(train_indices)}")
    print(f"test graphs:  {len(test_indices)}")

    exp_graphs = graphs
    if use_angular:
        exp_graphs = [add_angular_features(g) for g in graphs]

    train_graphs = [exp_graphs[i] for i in train_indices]
    test_graphs = [exp_graphs[i] for i in test_indices]

    train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

    # FIX 2: Dynamic dimension tracking explicitly prevents PyTorch metric mismatch errors.
    node_dim = train_graphs[0].x.shape[1]
    edge_dim = train_graphs[0].edge_attr.shape[1]

    model = ShockGNN(node_dim=node_dim, edge_dim=edge_dim, hidden_dim=64).to(device)
    model = train_model(model, train_loader, test_loader, use_augmentation=use_augmentation)

    metrics = evaluate_metrics(model, test_loader)
    rot_consistency_err = compute_rotation_consistency_error(model, test_loader)
    metrics["rot_consistency_err"] = rot_consistency_err

    print(f"\nMAE:            {metrics['mae']:.6f} A")
    print(f"RMSE:           {metrics['rmse']:.6f} A")
    print(f"Mean pred:      {metrics['mean_pred']:.6f} A")
    print(f"Mean true:      {metrics['mean_actual']:.6f} A")
    print(f"Max pred:       {metrics['max_pred']:.6f} A")
    print(f"Max true:       {metrics['max_actual']:.6f} A")
    print(f"Rot Consistency:{metrics['rot_consistency_err']:.6f}")

    return metrics


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--test", default="B_suite", choices=["B_suite", "B0", "B1", "B2", "B3"])
    args, unknown = parser.parse_known_args()

    structures = get_structures()
    graphs = build_dataset(structures)

    rotation_sanity_check(graphs[0])

    train_idx, test_idx = direction_split(graphs)
    results = {}

    if args.test in ["B_suite", "B0"]:
        results["B0"] = run_b_experiment("B0 — HELD-OUT DIRECTION (BASELINE)", graphs, train_idx, test_idx, False, False)

    if args.test in ["B_suite", "B1"]:
        results["B1"] = run_b_experiment("B1 — ROTATION AUGMENTATION", graphs, train_idx, test_idx, True, False)

    if args.test in ["B_suite", "B2"]:
        results["B2"] = run_b_experiment("B2 — EXPLICIT ANGLE", graphs, train_idx, test_idx, False, True)

    if args.test in ["B_suite", "B3"]:
        results["B3"] = run_b_experiment("B3 — ANGLE + ROTATION AUGMENTATION", graphs, train_idx, test_idx, True, True)

    print("\n" + "=" * 70)
    print("FINAL DIRECTIONAL GENERALIZATION SUMMARY")
    print("=" * 70)
    print(f"{'Exp':<8}{'Aug':<8}{'Angle':<8}{'MAE (A)':<12}{'RMSE (A)':<12}{'Rel MAE':<12}{'Rot Consist Err':<16}")
    print("-" * 70)

    b0_mae = results["B0"]["mae"] if "B0" in results else None

    for exp_id, res in results.items():
        aug_str = "ON" if "1" in exp_id or "3" in exp_id else "OFF"
        ang_str = "ON" if "2" in exp_id or "3" in exp_id else "OFF"

        rel_str = "0.00%"
        if b0_mae is not None and exp_id != "B0":
            rel_change = (res["mae"] - b0_mae) / b0_mae * 100
            rel_str = f"{rel_change:+.2f}%"

        print(f"{exp_id:<8}{aug_str:<8}{ang_str:<8}{res['mae']:<12.6f}{res['rmse']:<12.6f}{rel_str:<12}{res['rot_consistency_err']:<16.6f}")


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# ANORTHITE DYNAMICS — ROTATION / EQUIVARIANCE STRESS TEST
#
# Tests the already-trained Variant C model:
#
#   C = species + velocity + force
#
# NO TRAINING.
# NO TEST SET.
# NO CHECKPOINT SELECTION.
#
# Measures:
#
# 1. Original validation RMSE
#
# 2. Rotated validation RMSE:
#       RMSE[f(RX), R y]
#
# 3. Equivariance consistency error:
#       RMSE[f(RX), R f(X)]
#
# Uses multiple fixed random SO(3) rotations.
# ============================================================

import os
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import NNConv


# ============================================================
# Configuration
# ============================================================

ROOT = "/content/processed_production"
SPLIT_FILE = os.path.join(ROOT, "splits.json")

CHECKPOINT = (
    "/content/abc_outputs/"
    "best_variant_C.pt"
)

OUTPUT_FILE = (
    "/content/abc_outputs/"
    "rotation_stress_test_C.json"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

SEED = 318
ROTATION_SEED = 8675309

BATCH_SIZE = 48
HIDDEN = 64
CUTOFF = 3.5

# 10 rotations × 6000 validation graphs.
# No training, so this should still be relatively cheap.
N_ROTATIONS = 10

USE_AMP = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print("Checkpoint:", CHECKPOINT)

if not os.path.isfile(CHECKPOINT):
    raise FileNotFoundError(
        f"C checkpoint not found:\n"
        f"{CHECKPOINT}"
    )


# ============================================================
# Read split manifest
# ============================================================

with open(SPLIT_FILE, "r") as f:
    split_info = json.load(f)

train_runs = split_info["splits"]["train"]
val_runs = split_info["splits"]["val"]
test_runs = split_info["splits"]["test"]

assert len(train_runs) == 12
assert len(val_runs) == 12
assert len(test_runs) == 12

print()
print("Train trajectories:", len(train_runs))
print("Val trajectories:  ", len(val_runs))
print(
    "Test trajectories: ",
    len(test_runs),
    "(manifest only)"
)


# ============================================================
# Confirm test remains physically absent
# ============================================================

test_present = []

for entry in test_runs:

    path = os.path.join(
        ROOT,
        entry["run_id"]
    )

    if os.path.exists(path):
        test_present.append(
            entry["run_id"]
        )

if test_present:

    raise RuntimeError(
        "TEST DATA FOUND IN RUNTIME:\n"
        + "\n".join(test_present)
    )

print(
    "Physical test trajectories present: 0"
)

print(
    "TEST SET REMAINS SEALED."
)


# ============================================================
# Resolve paths
#
# Handles Windows paths stored in splits.json.
# ============================================================

def get_sample_paths(entries):

    paths = []

    for entry in entries:

        relative_sample_dir = (
            entry["sample_dir"]
            .replace("\\", "/")
        )

        sample_dir = os.path.join(
            ROOT,
            relative_sample_dir
        )

        if not os.path.isdir(sample_dir):

            raise FileNotFoundError(
                f"{entry['run_id']}: "
                f"directory not found:\n"
                f"{sample_dir}"
            )

        files = sorted(
            os.path.join(
                sample_dir,
                f
            )
            for f in os.listdir(
                sample_dir
            )
            if (
                f.startswith("sample_")
                and f.endswith(".npz")
            )
        )

        expected = entry["num_samples"]

        if len(files) != expected:

            raise RuntimeError(
                f"{entry['run_id']}: "
                f"expected {expected}, "
                f"got {len(files)}"
            )

        paths.extend(files)

    return paths


train_paths = get_sample_paths(
    train_runs
)

val_paths = get_sample_paths(
    val_runs
)

assert len(train_paths) == 6000
assert len(val_paths) == 6000

print()
print(
    "Train samples:",
    len(train_paths)
)

print(
    "Val samples:  ",
    len(val_paths)
)


# ============================================================
# Compute normalization from TRAIN ONLY
#
# This reproduces the A/B/C normalization.
# ============================================================

def compute_normalization(paths):

    vel_sum = np.zeros(
        3,
        dtype=np.float64
    )

    vel_sq_sum = np.zeros(
        3,
        dtype=np.float64
    )

    force_sum = np.zeros(
        3,
        dtype=np.float64
    )

    force_sq_sum = np.zeros(
        3,
        dtype=np.float64
    )

    n_atoms = 0

    start = time.time()

    for i, path in enumerate(paths):

        with np.load(path) as d:

            velocity = np.stack(
                [
                    d["vx_t"],
                    d["vy_t"],
                    d["vz_t"]
                ],
                axis=1
            ).astype(np.float64)

            force = np.stack(
                [
                    d["fx_t"],
                    d["fy_t"],
                    d["fz_t"]
                ],
                axis=1
            ).astype(np.float64)

        vel_sum += velocity.sum(
            axis=0
        )

        vel_sq_sum += (
            velocity ** 2
        ).sum(
            axis=0
        )

        force_sum += force.sum(
            axis=0
        )

        force_sq_sum += (
            force ** 2
        ).sum(
            axis=0
        )

        n_atoms += velocity.shape[0]

        if (i + 1) % 1000 == 0:

            print(
                f"Normalization: "
                f"{i+1}/{len(paths)}"
            )

    vel_mean = (
        vel_sum / n_atoms
    )

    force_mean = (
        force_sum / n_atoms
    )

    vel_var = (
        vel_sq_sum / n_atoms
        - vel_mean ** 2
    )

    force_var = (
        force_sq_sum / n_atoms
        - force_mean ** 2
    )

    vel_std = np.sqrt(
        np.maximum(
            vel_var,
            1e-12
        )
    )

    force_std = np.sqrt(
        np.maximum(
            force_var,
            1e-12
        )
    )

    print(
        f"Normalization computed in "
        f"{time.time()-start:.1f}s"
    )

    return {
        "velocity_mean":
            vel_mean.astype(
                np.float32
            ),

        "velocity_std":
            vel_std.astype(
                np.float32
            ),

        "force_mean":
            force_mean.astype(
                np.float32
            ),

        "force_std":
            force_std.astype(
                np.float32
            )
    }


norm = compute_normalization(
    train_paths
)

print()
print("TRAIN-ONLY NORMALIZATION")

print(
    "Velocity mean:",
    norm["velocity_mean"]
)

print(
    "Velocity std: ",
    norm["velocity_std"]
)

print(
    "Force mean:   ",
    norm["force_mean"]
)

print(
    "Force std:    ",
    norm["force_std"]
)


# ============================================================
# IMPORTANT NORMALIZATION NOTE
#
# We rotate PHYSICAL velocity/force vectors first,
# then apply the SAME training-set normalization.
#
# This correctly asks:
#
# "What happens if the physical system is globally rotated
#  while the trained model remains unchanged?"
#
# We do NOT rotate already-standardized vectors.
# ============================================================


# ============================================================
# Cache raw validation data ONCE
# ============================================================

class RawGraph:

    __slots__ = (
        "species",
        "velocity",
        "force",
        "edge_index",
        "edge_vector",
        "edge_distance",
        "target"
    )

    def __init__(
        self,
        species,
        velocity,
        force,
        edge_index,
        edge_vector,
        edge_distance,
        target
    ):

        self.species = species
        self.velocity = velocity
        self.force = force
        self.edge_index = edge_index
        self.edge_vector = edge_vector
        self.edge_distance = edge_distance
        self.target = target


def load_validation_cache(paths):

    cache = []

    start = time.time()

    for i, path in enumerate(paths):

        with np.load(path) as d:

            species = (
                d["atom_types"]
                .astype(np.float32)
                [:, None]
            )

            velocity = np.stack(
                [
                    d["vx_t"],
                    d["vy_t"],
                    d["vz_t"]
                ],
                axis=1
            ).astype(np.float32)

            force = np.stack(
                [
                    d["fx_t"],
                    d["fy_t"],
                    d["fz_t"]
                ],
                axis=1
            ).astype(np.float32)

            src = d[
                "edge_src"
            ].astype(np.int64)

            dst = d[
                "edge_dst"
            ].astype(np.int64)

            edge_index = np.stack(
                [src, dst],
                axis=0
            )

            edge_vector = d[
                "edge_vector"
            ].astype(np.float32)

            edge_distance = d[
                "edge_distance"
            ].astype(np.float32)

            target = d[
                "non_affine_displacement"
            ].astype(np.float32)

        cache.append(
            RawGraph(
                species,
                velocity,
                force,
                edge_index,
                edge_vector,
                edge_distance,
                target
            )
        )

        if (i + 1) % 1000 == 0:

            print(
                f"Val cache: "
                f"{i+1}/{len(paths)}"
            )

    print(
        f"Validation cache loaded "
        f"in {time.time()-start:.1f}s"
    )

    return cache


val_cache = load_validation_cache(
    val_paths
)

print()
print(
    "All 6000 validation graphs "
    "cached in RAM."
)


# ============================================================
# Generate proper random SO(3) rotation
#
# QR decomposition + determinant correction.
# ============================================================

def random_rotation_matrix(rng):

    A = rng.normal(
        size=(3, 3)
    )

    Q, R = np.linalg.qr(A)

    # Resolve QR sign ambiguity.
    signs = np.sign(
        np.diag(R)
    )

    signs[
        signs == 0
    ] = 1.0

    Q = Q @ np.diag(
        signs
    )

    # Force determinant +1:
    # proper rotation, not reflection.
    if np.linalg.det(Q) < 0:

        Q[:, 0] *= -1.0

    return Q.astype(
        np.float32
    )


rng = np.random.default_rng(
    ROTATION_SEED
)

rotations = [
    random_rotation_matrix(rng)
    for _ in range(N_ROTATIONS)
]

print()
print(
    f"Generated "
    f"{len(rotations)} "
    f"proper SO(3) rotations."
)

for i, R in enumerate(
    rotations[:3]
):

    print(
        f"Rotation {i+1} "
        f"det={np.linalg.det(R):.8f}"
    )


# ============================================================
# Dataset
#
# rotation=None:
#     ordinary validation graph
#
# rotation=R:
#     globally rotate:
#       velocity
#       force
#       edge vector
#       target displacement
#
# Species and distances remain unchanged.
# ============================================================

class RotationDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        cache,
        norm,
        rotation=None
    ):

        self.cache = cache

        self.rotation = (
            None
            if rotation is None
            else torch.tensor(
                rotation,
                dtype=torch.float32
            )
        )

        self.vel_mean = torch.tensor(
            norm["velocity_mean"],
            dtype=torch.float32
        )

        self.vel_std = torch.tensor(
            norm["velocity_std"],
            dtype=torch.float32
        )

        self.force_mean = torch.tensor(
            norm["force_mean"],
            dtype=torch.float32
        )

        self.force_std = torch.tensor(
            norm["force_std"],
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.cache)

    def __getitem__(
        self,
        idx
    ):

        g = self.cache[idx]

        species = torch.from_numpy(
            g.species
        )

        velocity = torch.from_numpy(
            g.velocity
        )

        force = torch.from_numpy(
            g.force
        )

        edge_vector = torch.from_numpy(
            g.edge_vector
        )

        target = torch.from_numpy(
            g.target
        )

        edge_distance = torch.from_numpy(
            g.edge_distance
        )

        edge_index = torch.from_numpy(
            g.edge_index
        )

        # --------------------------------------------
        # Row-vector convention:
        #
        # x' = x @ R.T
        # --------------------------------------------

        if self.rotation is not None:

            RT = self.rotation.T

            velocity = (
                velocity @ RT
            )

            force = (
                force @ RT
            )

            edge_vector = (
                edge_vector @ RT
            )

            target = (
                target @ RT
            )

        # --------------------------------------------
        # Normalize AFTER physical rotation using
        # original TRAIN statistics.
        # --------------------------------------------

        velocity = (
            velocity
            - self.vel_mean
        ) / self.vel_std

        force = (
            force
            - self.force_mean
        ) / self.force_std

        # --------------------------------------------
        # Reconstruct edge features.
        # --------------------------------------------

        direction = (
            edge_vector
            / torch.clamp(
                edge_distance[:, None],
                min=1e-8
            )
        )

        edge_attr = torch.cat(
            [
                direction,
                edge_distance[:, None]
                / CUTOFF
            ],
            dim=1
        )

        # Variant C:
        # species + velocity + force
        x = torch.cat(
            [
                species,
                velocity,
                force
            ],
            dim=1
        )

        return Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=target
        )


# ============================================================
# Model — EXACT same architecture as A/B/C
# ============================================================

class DynamicsGNN(nn.Module):

    def __init__(
        self,
        in_dim=7
    ):

        super().__init__()

        self.input = nn.Linear(
            in_dim,
            HIDDEN
        )

        edge_net1 = nn.Sequential(
            nn.Linear(4, 64),
            nn.SiLU(),
            nn.Linear(
                64,
                HIDDEN * HIDDEN
            )
        )

        self.conv1 = NNConv(
            HIDDEN,
            HIDDEN,
            edge_net1,
            aggr="mean"
        )

        edge_net2 = nn.Sequential(
            nn.Linear(4, 64),
            nn.SiLU(),
            nn.Linear(
                64,
                HIDDEN * HIDDEN
            )
        )

        self.conv2 = NNConv(
            HIDDEN,
            HIDDEN,
            edge_net2,
            aggr="mean"
        )

        self.head = nn.Sequential(
            nn.Linear(
                HIDDEN,
                HIDDEN
            ),
            nn.SiLU(),
            nn.Linear(
                HIDDEN,
                3
            )
        )

    def forward(
        self,
        data
    ):

        x = self.input(
            data.x
        )

        x = torch.nn.functional.silu(
            x
        )

        residual = x

        x = self.conv1(
            x,
            data.edge_index,
            data.edge_attr
        )

        x = torch.nn.functional.silu(
            x
        )

        x = self.conv2(
            x,
            data.edge_index,
            data.edge_attr
        )

        x = torch.nn.functional.silu(
            x + residual
        )

        return self.head(x)


# ============================================================
# Load frozen C checkpoint
# ============================================================

model = DynamicsGNN(
    in_dim=7
).to(DEVICE)

model.load_state_dict(
    torch.load(
        CHECKPOINT,
        map_location=DEVICE,
        weights_only=True
    )
)

model.eval()

print()
print(
    "Variant C checkpoint loaded."
)

print(
    "Parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)


# ============================================================
# Predict entire dataset
# ============================================================

@torch.no_grad()
def predict_dataset(
    dataset
):

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(
            DEVICE.type == "cuda"
        )
    )

    predictions = []
    targets = []

    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):

            pred = model(
                batch
            )

        predictions.append(
            pred.float().cpu()
        )

        targets.append(
            batch.y.float().cpu()
        )

    return (
        torch.cat(
            predictions,
            dim=0
        ),
        torch.cat(
            targets,
            dim=0
        )
    )


# ============================================================
# Metric helper
# ============================================================

def metrics(
    pred,
    target
):

    error = (
        pred - target
    )

    mse = torch.mean(
        error ** 2
    ).item()

    rmse = np.sqrt(
        mse
    )

    mae = torch.mean(
        torch.abs(error)
    ).item()

    return {
        "rmse": float(rmse),
        "mae": float(mae)
    }


# ============================================================
# Original validation evaluation
# ============================================================

print()
print("=" * 72)
print("ORIGINAL VALIDATION")
print("=" * 72)

original_dataset = RotationDataset(
    val_cache,
    norm,
    rotation=None
)

original_pred, original_target = (
    predict_dataset(
        original_dataset
    )
)

original_metrics = metrics(
    original_pred,
    original_target
)

print(
    f"Original RMSE: "
    f"{original_metrics['rmse']:.8f} Å"
)

print(
    f"Original MAE:  "
    f"{original_metrics['mae']:.8f} Å"
)

# Sanity check against previous A/B/C result.
EXPECTED_C_RMSE = 0.00305815

difference = abs(
    original_metrics["rmse"]
    - EXPECTED_C_RMSE
)

print(
    f"Difference from prior "
    f"C RMSE: {difference:.8e} Å"
)

if difference > 5e-5:

    raise RuntimeError(
        "Original validation RMSE does not "
        "reproduce the saved C result closely "
        "enough. Stop before interpreting "
        "rotation results."
    )

print(
    "Checkpoint reproduction: PASS"
)


# ============================================================
# Rotation tests
# ============================================================

print()
print("=" * 72)
print("ROTATION STRESS TEST")
print("=" * 72)

rotation_results = []

for rotation_id, R in enumerate(
    rotations,
    start=1
):

    start = time.time()

    rotated_dataset = RotationDataset(
        val_cache,
        norm,
        rotation=R
    )

    (
        rotated_pred,
        rotated_target
    ) = predict_dataset(
        rotated_dataset
    )

    # --------------------------------------------
    # Metric 1:
    #
    # Physical prediction accuracy after rotating
    # the complete system.
    #
    # RMSE[f(RX), Ry]
    # --------------------------------------------

    rotated_accuracy = metrics(
        rotated_pred,
        rotated_target
    )

    # --------------------------------------------
    # Metric 2:
    #
    # Strict equivariance consistency.
    #
    # original_pred contains vectors p.
    #
    # Expected rotated prediction:
    #       p' = p @ R.T
    #
    # Compare:
    #       f(RX)
    # against
    #       R f(X)
    # --------------------------------------------

    R_tensor = torch.tensor(
        R,
        dtype=torch.float32
    )

    expected_rotated_pred = (
        original_pred
        @ R_tensor.T
    )

    equivariance = metrics(
        rotated_pred,
        expected_rotated_pred
    )

    degradation = (
        rotated_accuracy["rmse"]
        / original_metrics["rmse"]
    )

    result = {
        "rotation_id":
            rotation_id,

        "rotation_matrix":
            R.tolist(),

        "determinant":
            float(
                np.linalg.det(R)
            ),

        "rotated_target_rmse":
            rotated_accuracy["rmse"],

        "rotated_target_mae":
            rotated_accuracy["mae"],

        "equivariance_rmse":
            equivariance["rmse"],

        "equivariance_mae":
            equivariance["mae"],

        "accuracy_degradation_factor":
            float(degradation)
    }

    rotation_results.append(
        result
    )

    print(
        f"Rotation {rotation_id:02d} | "
        f"target RMSE "
        f"{rotated_accuracy['rmse']:.8f} Å | "
        f"equiv RMSE "
        f"{equivariance['rmse']:.8f} Å | "
        f"degradation "
        f"{degradation:.2f}x | "
        f"{time.time()-start:.1f}s"
    )


# ============================================================
# Aggregate statistics
# ============================================================

rotated_rmse = np.array(
    [
        r["rotated_target_rmse"]
        for r in rotation_results
    ],
    dtype=np.float64
)

rotated_mae = np.array(
    [
        r["rotated_target_mae"]
        for r in rotation_results
    ],
    dtype=np.float64
)

equiv_rmse = np.array(
    [
        r["equivariance_rmse"]
        for r in rotation_results
    ],
    dtype=np.float64
)

equiv_mae = np.array(
    [
        r["equivariance_mae"]
        for r in rotation_results
    ],
    dtype=np.float64
)

degradation = (
    rotated_rmse
    / original_metrics["rmse"]
)


summary = {

    "original_validation": {
        "rmse":
            original_metrics["rmse"],

        "mae":
            original_metrics["mae"]
    },

    "rotated_target_rmse": {
        "mean":
            float(
                rotated_rmse.mean()
            ),

        "std":
            float(
                rotated_rmse.std(
                    ddof=1
                )
            ),

        "min":
            float(
                rotated_rmse.min()
            ),

        "max":
            float(
                rotated_rmse.max()
            )
    },

    "rotated_target_mae": {
        "mean":
            float(
                rotated_mae.mean()
            ),

        "std":
            float(
                rotated_mae.std(
                    ddof=1
                )
            )
    },

    "equivariance_rmse": {
        "mean":
            float(
                equiv_rmse.mean()
            ),

        "std":
            float(
                equiv_rmse.std(
                    ddof=1
                )
            ),

        "min":
            float(
                equiv_rmse.min()
            ),

        "max":
            float(
                equiv_rmse.max()
            )
    },

    "equivariance_mae": {
        "mean":
            float(
                equiv_mae.mean()
            ),

        "std":
            float(
                equiv_mae.std(
                    ddof=1
                )
            )
    },

    "accuracy_degradation_factor": {
        "mean":
            float(
                degradation.mean()
            ),

        "std":
            float(
                degradation.std(
                    ddof=1
                )
            ),

        "min":
            float(
                degradation.min()
            ),

        "max":
            float(
                degradation.max()
            )
    }
}


# ============================================================
# Print final result
# ============================================================

print()
print("=" * 72)
print("FINAL ROTATION SUMMARY")
print("=" * 72)

print(
    f"Original RMSE: "
    f"{original_metrics['rmse']:.8f} Å"
)

print()

print(
    "Rotated-target RMSE:"
)

print(
    f"  mean = "
    f"{rotated_rmse.mean():.8f} Å"
)

print(
    f"  std  = "
    f"{rotated_rmse.std(ddof=1):.8f} Å"
)

print(
    f"  min  = "
    f"{rotated_rmse.min():.8f} Å"
)

print(
    f"  max  = "
    f"{rotated_rmse.max():.8f} Å"
)

print()

print(
    "Equivariance RMSE:"
)

print(
    f"  mean = "
    f"{equiv_rmse.mean():.8f} Å"
)

print(
    f"  std  = "
    f"{equiv_rmse.std(ddof=1):.8f} Å"
)

print(
    f"  min  = "
    f"{equiv_rmse.min():.8f} Å"
)

print(
    f"  max  = "
    f"{equiv_rmse.max():.8f} Å"
)

print()

print(
    "Accuracy degradation factor:"
)

print(
    f"  mean = "
    f"{degradation.mean():.3f}x"
)

print(
    f"  min  = "
    f"{degradation.min():.3f}x"
)

print(
    f"  max  = "
    f"{degradation.max():.3f}x"
)


# ============================================================
# Save machine-readable result
# ============================================================

output = {

    "experiment":
        "variant_C_rotation_stress_test",

    "checkpoint":
        CHECKPOINT,

    "split":
        "validation_seed_22002",

    "test_set_used":
        False,

    "rotation_seed":
        ROTATION_SEED,

    "n_rotations":
        N_ROTATIONS,

    "summary":
        summary,

    "rotations":
        rotation_results
}


with open(
    OUTPUT_FILE,
    "w"
) as f:

    json.dump(
        output,
        f,
        indent=2
    )

print()
print(
    "Saved:",
    OUTPUT_FILE
)

print()
print(
    "TEST SET HAS NOT BEEN "
    "LOADED OR EVALUATED."
)

In [ ]:
# ============================================================
# ANORTHITE DYNAMICS
# E(3)-EQUIVARIANT A/B/C STATE REPRESENTATION EXPERIMENT
# + AUTOMATIC SO(3) ROTATION STRESS TEST
#
# Environment verified:
#   PyTorch 2.11.0
#   e3nn 0.6.0
#
# REPRESENTATIONS
# ------------------------------------------------------------
# A = species
# B = species + velocity
# C = species + velocity + force
#
# E(3) interpretation:
#
#   species  -> scalar 0e
#   velocity -> vector 1o
#   force    -> vector 1o
#   edge r   -> spherical harmonics
#   target   -> vector 1o
#
# TRAIN: seed 11001 — 6000 samples
# VAL:   seed 22002 — 6000 samples
# TEST:  PHYSICALLY ABSENT / NEVER LOADED
#
# IMPORTANT:
#   - train-only normalization
#   - same trajectory split as NNConv
#   - cached RAM dataset
#   - best checkpoint chosen ONLY by validation MSE
#   - no test-set access
#   - no rotation augmentation
#   - rotation test occurs AFTER training
# ============================================================


# ============================================================
# Install
# ============================================================

%pip -q install e3nn==0.6.0 torch-geometric


# ============================================================
# Imports
# ============================================================

import os
import json
import math
import random
import time
import shutil

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from e3nn import o3


# ============================================================
# Configuration
# ============================================================

ROOT = "/content/processed_production"

SPLIT_FILE = os.path.join(
    ROOT,
    "splits.json"
)

OUTPUT_DIR = "/content/e3nn_abc_outputs"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

SEED = 318
ROTATION_SEED = 8675309

BATCH_SIZE = 48

EPOCHS = 200

LR = 1e-3
WEIGHT_DECAY = 0.0

CUTOFF = 3.5

VAL_EVERY = 5
PATIENCE_EVALS = 6

N_ROTATIONS = 10

# ------------------------------------------------------------
# Equivariant model size
#
# Hidden irreps contain:
#
#   32 scalar-even channels
#   16 vector-odd channels
#
# This is intentionally modest for a T4.
# ------------------------------------------------------------

HIDDEN_IRREPS = o3.Irreps(
    "32x0e + 16x1o"
)

# Edge spherical harmonics up through l=2.
EDGE_IRREPS = o3.Irreps.spherical_harmonics(
    lmax=2
)

# Output is a polar vector.
OUTPUT_IRREPS = o3.Irreps(
    "1x1o"
)

# Number of radial basis functions used to generate
# tensor-product weights.
NUM_RADIAL = 8

RADIAL_HIDDEN = 64


# ============================================================
# Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )

print("Device:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "PyTorch:",
    torch.__version__
)

import e3nn

print(
    "e3nn:",
    e3nn.__version__
)

print(
    "Hidden irreps:",
    HIDDEN_IRREPS
)

print(
    "Edge irreps:",
    EDGE_IRREPS
)

print(
    "Output irreps:",
    OUTPUT_IRREPS
)

print()
print(
    "AMP intentionally disabled "
    "for equivariant tensor products."
)


# ============================================================
# Read split manifest
# ============================================================

with open(
    SPLIT_FILE,
    "r"
) as f:

    split_info = json.load(f)


train_runs = split_info[
    "splits"
]["train"]

val_runs = split_info[
    "splits"
]["val"]

test_runs = split_info[
    "splits"
]["test"]


assert len(train_runs) == 12
assert len(val_runs) == 12
assert len(test_runs) == 12


print()
print(
    "Train trajectories:",
    len(train_runs)
)

print(
    "Val trajectories:  ",
    len(val_runs)
)

print(
    "Test trajectories: ",
    len(test_runs),
    "(manifest only)"
)


# ============================================================
# Verify TEST is physically absent
# ============================================================

test_present = []

for entry in test_runs:

    test_dir = os.path.join(
        ROOT,
        entry["run_id"]
    )

    if os.path.exists(
        test_dir
    ):

        test_present.append(
            entry["run_id"]
        )


if test_present:

    raise RuntimeError(
        "TEST DATA IS PRESENT IN RUNTIME:\n"
        + "\n".join(
            test_present
        )
    )


print(
    "Physical test trajectories present: 0"
)

print(
    "TEST SET IS SEALED."
)


# ============================================================
# Resolve sample paths
#
# Handles Windows backslashes in splits.json.
# ============================================================

def get_sample_paths(entries):

    paths = []

    for entry in entries:

        relative_sample_dir = (
            entry["sample_dir"]
            .replace("\\", "/")
        )

        sample_dir = os.path.join(
            ROOT,
            relative_sample_dir
        )

        if not os.path.isdir(
            sample_dir
        ):

            raise FileNotFoundError(
                f"{entry['run_id']}: "
                f"sample directory not found:\n"
                f"{sample_dir}"
            )

        files = sorted(
            os.path.join(
                sample_dir,
                f
            )
            for f in os.listdir(
                sample_dir
            )
            if (
                f.startswith(
                    "sample_"
                )
                and f.endswith(
                    ".npz"
                )
            )
        )

        expected = entry[
            "num_samples"
        ]

        if len(files) != expected:

            raise RuntimeError(
                f"{entry['run_id']}: "
                f"expected {expected}, "
                f"got {len(files)}"
            )

        paths.extend(
            files
        )

    return paths


train_paths = get_sample_paths(
    train_runs
)

val_paths = get_sample_paths(
    val_runs
)


assert len(train_paths) == 6000
assert len(val_paths) == 6000


print()
print(
    "Train samples:",
    len(train_paths)
)

print(
    "Val samples:  ",
    len(val_paths)
)

print(
    "First train sample:",
    train_paths[0]
)


# ============================================================
# Cached graph representation
# ============================================================

class CachedGraph:

    __slots__ = (
        "species",
        "velocity",
        "force",
        "edge_index",
        "edge_vector",
        "edge_distance",
        "y"
    )

    def __init__(
        self,
        species,
        velocity,
        force,
        edge_index,
        edge_vector,
        edge_distance,
        y
    ):

        self.species = species
        self.velocity = velocity
        self.force = force

        self.edge_index = edge_index

        self.edge_vector = edge_vector
        self.edge_distance = edge_distance

        self.y = y


# ============================================================
# Load TRAIN once
# + calculate TRAIN-ONLY normalization
# ============================================================

def load_train_cache(paths):

    cache = []

    vel_sum = np.zeros(
        3,
        dtype=np.float64
    )

    vel_sq_sum = np.zeros(
        3,
        dtype=np.float64
    )

    force_sum = np.zeros(
        3,
        dtype=np.float64
    )

    force_sq_sum = np.zeros(
        3,
        dtype=np.float64
    )

    n_atoms = 0

    start = time.time()

    for i, path in enumerate(
        paths
    ):

        with np.load(
            path
        ) as d:

            species = (
                d["atom_types"]
                .astype(
                    np.float32
                )[:, None]
            )

            velocity = np.stack(
                [
                    d["vx_t"],
                    d["vy_t"],
                    d["vz_t"]
                ],
                axis=1
            ).astype(
                np.float32
            )

            force = np.stack(
                [
                    d["fx_t"],
                    d["fy_t"],
                    d["fz_t"]
                ],
                axis=1
            ).astype(
                np.float32
            )

            src = d[
                "edge_src"
            ].astype(
                np.int64
            )

            dst = d[
                "edge_dst"
            ].astype(
                np.int64
            )

            edge_index = np.stack(
                [
                    src,
                    dst
                ],
                axis=0
            )

            edge_vector = (
                d["edge_vector"]
                .astype(
                    np.float32
                )
            )

            edge_distance = (
                d["edge_distance"]
                .astype(
                    np.float32
                )
            )

            y = (
                d[
                    "non_affine_displacement"
                ]
                .astype(
                    np.float32
                )
            )

        # --------------------------------------------
        # Train-only normalization statistics
        # --------------------------------------------

        vel64 = velocity.astype(
            np.float64
        )

        force64 = force.astype(
            np.float64
        )

        vel_sum += vel64.sum(
            axis=0
        )

        vel_sq_sum += (
            vel64 ** 2
        ).sum(
            axis=0
        )

        force_sum += force64.sum(
            axis=0
        )

        force_sq_sum += (
            force64 ** 2
        ).sum(
            axis=0
        )

        n_atoms += (
            velocity.shape[0]
        )

        cache.append(
            CachedGraph(
                torch.from_numpy(
                    species
                ),
                torch.from_numpy(
                    velocity
                ),
                torch.from_numpy(
                    force
                ),
                torch.from_numpy(
                    edge_index
                ),
                torch.from_numpy(
                    edge_vector
                ),
                torch.from_numpy(
                    edge_distance
                ),
                torch.from_numpy(
                    y
                )
            )
        )

        if (
            (i + 1) % 1000
            == 0
        ):

            print(
                f"Train cache: "
                f"{i+1}/{len(paths)}"
            )


    vel_mean = (
        vel_sum
        / n_atoms
    )

    force_mean = (
        force_sum
        / n_atoms
    )


    vel_var = (
        vel_sq_sum
        / n_atoms
        - vel_mean ** 2
    )

    force_var = (
        force_sq_sum
        / n_atoms
        - force_mean ** 2
    )


    vel_std = np.sqrt(
        np.maximum(
            vel_var,
            1e-12
        )
    )

    force_std = np.sqrt(
        np.maximum(
            force_var,
            1e-12
        )
    )


    norm = {

        "velocity_mean":
            torch.tensor(
                vel_mean,
                dtype=torch.float32
            ),

        "velocity_std":
            torch.tensor(
                vel_std,
                dtype=torch.float32
            ),

        "force_mean":
            torch.tensor(
                force_mean,
                dtype=torch.float32
            ),

        "force_std":
            torch.tensor(
                force_std,
                dtype=torch.float32
            )
    }


    print(
        f"Train cache loaded in "
        f"{time.time()-start:.1f}s"
    )

    return (
        cache,
        norm
    )


train_cache, norm = (
    load_train_cache(
        train_paths
    )
)


# ============================================================
# IMPORTANT EQUIVARIANCE NORMALIZATION
# ============================================================
#
# A vector representation must not be normalized with
# three substantially different axis-dependent scales if
# exact rotational equivariance is desired.
#
# The old NNConv experiment used per-component std.
#
# Here we convert those train statistics into ONE scalar
# normalization scale per vector field:
#
#   sigma_v = sqrt(mean(sigma_x^2, sigma_y^2, sigma_z^2))
#
#   sigma_f = sqrt(mean(...))
#
# Mean vector is also removed.
#
# This preserves vector transformation structure.
# ============================================================

velocity_scale = torch.sqrt(
    torch.mean(
        norm[
            "velocity_std"
        ] ** 2
    )
)

force_scale = torch.sqrt(
    torch.mean(
        norm[
            "force_std"
        ] ** 2
    )
)


print()
print(
    "TRAIN-ONLY NORMALIZATION"
)

print(
    "Velocity mean:",
    norm[
        "velocity_mean"
    ].numpy()
)

print(
    "Original component std:",
    norm[
        "velocity_std"
    ].numpy()
)

print(
    "Equivariant velocity scale:",
    velocity_scale.item()
)

print()

print(
    "Force mean:",
    norm[
        "force_mean"
    ].numpy()
)

print(
    "Original component std:",
    norm[
        "force_std"
    ].numpy()
)

print(
    "Equivariant force scale:",
    force_scale.item()
)


# ============================================================
# Normalize TRAIN cache once
#
# IMPORTANT:
# Mean subtraction technically selects a frame if the
# empirical mean is nonzero.
#
# The measured means are extremely small relative to the
# velocity scale. We preserve the train-centering used by
# the baseline while using isotropic scaling.
# ============================================================

for g in train_cache:

    g.velocity = (
        g.velocity
        - norm[
            "velocity_mean"
        ]
    ) / velocity_scale

    g.force = (
        g.force
        - norm[
            "force_mean"
        ]
    ) / force_scale


# ============================================================
# Load VAL once using TRAIN statistics
# ============================================================

def load_val_cache(
    paths,
    norm,
    velocity_scale,
    force_scale
):

    cache = []

    start = time.time()

    for i, path in enumerate(
        paths
    ):

        with np.load(
            path
        ) as d:

            species = (
                d["atom_types"]
                .astype(
                    np.float32
                )[:, None]
            )

            velocity = np.stack(
                [
                    d["vx_t"],
                    d["vy_t"],
                    d["vz_t"]
                ],
                axis=1
            ).astype(
                np.float32
            )

            force = np.stack(
                [
                    d["fx_t"],
                    d["fy_t"],
                    d["fz_t"]
                ],
                axis=1
            ).astype(
                np.float32
            )

            src = d[
                "edge_src"
            ].astype(
                np.int64
            )

            dst = d[
                "edge_dst"
            ].astype(
                np.int64
            )

            edge_index = np.stack(
                [
                    src,
                    dst
                ],
                axis=0
            )

            edge_vector = (
                d["edge_vector"]
                .astype(
                    np.float32
                )
            )

            edge_distance = (
                d["edge_distance"]
                .astype(
                    np.float32
                )
            )

            y = (
                d[
                    "non_affine_displacement"
                ]
                .astype(
                    np.float32
                )
            )


        velocity = (
            torch.from_numpy(
                velocity
            )
            - norm[
                "velocity_mean"
            ]
        ) / velocity_scale


        force = (
            torch.from_numpy(
                force
            )
            - norm[
                "force_mean"
            ]
        ) / force_scale


        cache.append(
            CachedGraph(
                torch.from_numpy(
                    species
                ),
                velocity,
                force,
                torch.from_numpy(
                    edge_index
                ),
                torch.from_numpy(
                    edge_vector
                ),
                torch.from_numpy(
                    edge_distance
                ),
                torch.from_numpy(
                    y
                )
            )
        )


        if (
            (i + 1) % 1000
            == 0
        ):

            print(
                f"Val cache: "
                f"{i+1}/{len(paths)}"
            )


    print(
        f"Val cache loaded in "
        f"{time.time()-start:.1f}s"
    )

    return cache


val_cache = load_val_cache(
    val_paths,
    norm,
    velocity_scale,
    force_scale
)


print()
print(
    "All 12,000 train+val "
    "graphs cached in RAM."
)


# ============================================================
# Zero predictor baseline
# ============================================================

def zero_baseline(
    cache
):

    squared = 0.0
    absolute = 0.0

    n = 0

    for g in cache:

        squared += torch.sum(
            g.y ** 2
        ).item()

        absolute += torch.sum(
            torch.abs(
                g.y
            )
        ).item()

        n += g.y.numel()

    mse = squared / n

    return (
        math.sqrt(
            mse
        ),
        absolute / n
    )


zero_rmse, zero_mae = (
    zero_baseline(
        val_cache
    )
)


print()
print(
    "VAL ZERO-PREDICTOR BASELINE"
)

print(
    f"RMSE: "
    f"{zero_rmse:.8f} Å"
)

print(
    f"MAE:  "
    f"{zero_mae:.8f} Å"
)


# ============================================================
# Dataset
# ============================================================

class EquivariantDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        cache,
        variant
    ):

        assert variant in [
            "A",
            "B",
            "C"
        ]

        self.cache = cache
        self.variant = variant


    def __len__(
        self
    ):

        return len(
            self.cache
        )


    def __getitem__(self, idx):

        g = self.cache[idx]

        return Data(
            species=g.species,
            velocity=g.velocity,
            force=g.force,
            edge_index=g.edge_index,
            edge_vector=g.edge_vector,
            edge_distance=g.edge_distance,
            y=g.y,
            num_nodes=g.species.shape[0]
        )


# ============================================================
# Irreps for each representation
# ============================================================

def input_irreps_for_variant(
    variant
):

    if variant == "A":

        return o3.Irreps(
            "1x0e"
        )

    elif variant == "B":

        return o3.Irreps(
            "1x0e + 1x1o"
        )

    else:

        return o3.Irreps(
            "1x0e + 2x1o"
        )


def build_input_features(
    data,
    variant
):

    if variant == "A":

        return data.species

    elif variant == "B":

        return torch.cat(
            [
                data.species,
                data.velocity
            ],
            dim=-1
        )

    else:

        return torch.cat(
            [
                data.species,
                data.velocity,
                data.force
            ],
            dim=-1
        )


# ============================================================
# Smooth radial basis
# ============================================================

def radial_basis(
    distance,
    cutoff=CUTOFF,
    num_basis=NUM_RADIAL
):

    # Normalize distance to [0,1].
    x = (
        distance
        / cutoff
    )

    centers = torch.linspace(
        0.0,
        1.0,
        num_basis,
        device=distance.device,
        dtype=distance.dtype
    )

    width = (
        1.0
        / num_basis
    )

    basis = torch.exp(
        -0.5
        * (
            (
                x[:, None]
                - centers[None, :]
            )
            / width
        ) ** 2
    )

    # Cosine cutoff envelope.
    envelope = (
        0.5
        * (
            torch.cos(
                math.pi
                * torch.clamp(
                    x,
                    0.0,
                    1.0
                )
            )
            + 1.0
        )
    )

    envelope = (
        envelope
        * (
            x <= 1.0
        ).to(
            distance.dtype
        )
    )

    return (
        basis
        * envelope[:, None]
    )


# ============================================================
# Equivariant message-passing layer
# ============================================================

class EquivariantConv(
    nn.Module
):

    def __init__(
        self,
        irreps_in,
        irreps_out
    ):

        super().__init__()

        self.irreps_in = o3.Irreps(
            irreps_in
        )

        self.irreps_out = o3.Irreps(
            irreps_out
        )

        self.irreps_edge = (
            EDGE_IRREPS
        )

        # ----------------------------------------------------
        # FullyConnectedTensorProduct determines all allowed
        # equivariant couplings:
        #
        # node irreps × spherical harmonics
        #       -> output irreps
        #
        # Internal weights are supplied per edge by radial MLP.
        # ----------------------------------------------------

        self.tp = (
            o3.FullyConnectedTensorProduct(
                self.irreps_in,
                self.irreps_edge,
                self.irreps_out,
                shared_weights=False,
                internal_weights=False
            )
        )

        self.radial_mlp = (
            nn.Sequential(

                nn.Linear(
                    NUM_RADIAL,
                    RADIAL_HIDDEN
                ),

                nn.SiLU(),

                nn.Linear(
                    RADIAL_HIDDEN,
                    self.tp.weight_numel
                )
            )
        )


    def forward(
        self,
        x,
        edge_index,
        edge_vector,
        edge_distance
    ):

        src = edge_index[0]
        dst = edge_index[1]

        # ----------------------------------------------------
        # Spherical harmonics of edge directions.
        #
        # normalize=True means direction magnitude does not
        # enter angular features.
        #
        # Distances separately enter radial network.
        # ----------------------------------------------------

        edge_sh = o3.spherical_harmonics(
            self.irreps_edge,
            edge_vector,
            normalize=True,
            normalization="component"
        )

        radial = radial_basis(
            edge_distance
        )

        weights = self.radial_mlp(
            radial
        )

        messages = self.tp(
            x[src],
            edge_sh,
            weights
        )

        # Mean aggregation, matching NNConv experiment.
        out = torch.zeros(
            (
                x.shape[0],
                self.irreps_out.dim
            ),
            dtype=x.dtype,
            device=x.device
        )

        out.index_add_(
            0,
            dst,
            messages
        )

        degree = torch.zeros(
            x.shape[0],
            dtype=x.dtype,
            device=x.device
        )

        degree.index_add_(
            0,
            dst,
            torch.ones(
                dst.shape[0],
                dtype=x.dtype,
                device=x.device
            )
        )

        out = (
            out
            / degree.clamp(
                min=1.0
            )[:, None]
        )

        return out


# ============================================================
# Equivariant nonlinearity
#
# Scalars can pass through SiLU.
#
# Vectors cannot have componentwise SiLU because that would
# break equivariance.
#
# Instead:
#
#       v -> sigmoid(a ||v|| + b) v
#
# where the gate is a rotational scalar.
# ============================================================

class EquivariantActivation(
    nn.Module
):

    def __init__(
        self,
        irreps
    ):

        super().__init__()

        self.irreps = o3.Irreps(
            irreps
        )

        self.vector_gates = (
            nn.ParameterList()
        )

        self.vector_biases = (
            nn.ParameterList()
        )

        for mul, ir in self.irreps:

            if ir.l > 0:

                self.vector_gates.append(
                    nn.Parameter(
                        torch.ones(
                            mul
                        )
                    )
                )

                self.vector_biases.append(
                    nn.Parameter(
                        torch.zeros(
                            mul
                        )
                    )
                )


    def forward(
        self,
        x
    ):

        pieces = []

        offset = 0
        vector_index = 0

        for mul, ir in self.irreps:

            block_dim = (
                mul
                * ir.dim
            )

            block = x[
                :,
                offset:
                offset + block_dim
            ]

            block = block.reshape(
                x.shape[0],
                mul,
                ir.dim
            )

            if ir.l == 0:

                block = F.silu(
                    block
                )

            else:

                norms = torch.sqrt(
                    torch.sum(
                        block ** 2,
                        dim=-1
                    )
                    + 1e-8
                )

                gate = torch.sigmoid(
                    norms
                    * self.vector_gates[
                        vector_index
                    ][None, :]
                    + self.vector_biases[
                        vector_index
                    ][None, :]
                )

                block = (
                    block
                    * gate[:, :, None]
                )

                vector_index += 1

            pieces.append(
                block.reshape(
                    x.shape[0],
                    block_dim
                )
            )

            offset += block_dim

        return torch.cat(
            pieces,
            dim=-1
        )


# ============================================================
# Full E(3)-equivariant dynamics model
# ============================================================

class E3DynamicsGNN(
    nn.Module
):

    def __init__(
        self,
        variant
    ):

        super().__init__()

        self.variant = variant

        self.irreps_input = (
            input_irreps_for_variant(
                variant
            )
        )

        self.irreps_hidden = (
            HIDDEN_IRREPS
        )

        self.irreps_output = (
            OUTPUT_IRREPS
        )

        # Equivariant linear embedding.
        self.input_linear = o3.Linear(
            self.irreps_input,
            self.irreps_hidden
        )

        self.conv1 = EquivariantConv(
            self.irreps_hidden,
            self.irreps_hidden
        )

        self.act1 = EquivariantActivation(
            self.irreps_hidden
        )

        self.conv2 = EquivariantConv(
            self.irreps_hidden,
            self.irreps_hidden
        )

        self.act2 = EquivariantActivation(
            self.irreps_hidden
        )

        self.output_linear = o3.Linear(
            self.irreps_hidden,
            self.irreps_output
        )


    def forward(
        self,
        data
    ):

        x = build_input_features(
            data,
            self.variant
        )

        x = self.input_linear(
            x
        )

        residual = x

        x = self.conv1(
            x,
            data.edge_index,
            data.edge_vector,
            data.edge_distance
        )

        x = self.act1(
            x
        )

        x = self.conv2(
            x,
            data.edge_index,
            data.edge_vector,
            data.edge_distance
        )

        x = (
            x
            + residual
        )

        x = self.act2(
            x
        )

        return self.output_linear(
            x
        )


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()

    sum_squared = 0.0
    sum_absolute = 0.0

    n_values = 0

    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        pred = model(
            batch
        )

        error = (
            pred.float()
            - batch.y.float()
        )

        sum_squared += torch.sum(
            error ** 2
        ).item()

        sum_absolute += torch.sum(
            torch.abs(
                error
            )
        ).item()

        n_values += (
            error.numel()
        )

    mse = (
        sum_squared
        / n_values
    )

    return (
        mse,
        math.sqrt(
            mse
        ),
        sum_absolute
        / n_values
    )


# ============================================================
# Train one representation
# ============================================================

def train_variant(
    variant
):

    print()
    print(
        "=" * 72
    )

    print(
        f"E3 VARIANT {variant}"
    )

    print(
        "=" * 72
    )


    random.seed(
        SEED
    )

    np.random.seed(
        SEED
    )

    torch.manual_seed(
        SEED
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            SEED
        )


    train_dataset = (
        EquivariantDataset(
            train_cache,
            variant
        )
    )

    val_dataset = (
        EquivariantDataset(
            val_cache,
            variant
        )
    )


    generator = (
        torch.Generator()
    )

    generator.manual_seed(
        SEED
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        generator=generator,

        num_workers=0,

        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    val_loader = DataLoader(

        val_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    model = E3DynamicsGNN(
        variant
    ).to(
        DEVICE
    )


    parameter_count = sum(
        p.numel()
        for p in model.parameters()
    )


    print(
        "Input irreps:",
        model.irreps_input
    )

    print(
        "Hidden irreps:",
        model.irreps_hidden
    )

    print(
        "Output irreps:",
        model.irreps_output
    )

    print(
        "Parameters:",
        parameter_count
    )


    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=LR,

        weight_decay=WEIGHT_DECAY
    )


    criterion = nn.MSELoss()


    best_val = float(
        "inf"
    )

    best_epoch = 0

    evals_without_improvement = 0


    checkpoint = os.path.join(

        OUTPUT_DIR,

        f"best_e3_variant_{variant}.pt"
    )


    history = []

    variant_start = time.time()


    for epoch in range(
        1,
        EPOCHS + 1
    ):

        epoch_start = time.time()

        model.train()

        train_squared = 0.0
        train_values = 0


        for batch in train_loader:

            batch = batch.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            pred = model(
                batch
            )

            loss = criterion(
                pred,
                batch.y
            )

            if not torch.isfinite(
                loss
            ):

                raise RuntimeError(
                    f"Non-finite loss in "
                    f"variant {variant}, "
                    f"epoch {epoch}"
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=10.0
            )

            optimizer.step()


            error = (
                pred.detach()
                - batch.y
            )

            train_squared += torch.sum(
                error ** 2
            ).item()

            train_values += (
                error.numel()
            )


        train_mse = (
            train_squared
            / train_values
        )

        train_rmse = math.sqrt(
            train_mse
        )


        do_validation = (

            epoch == 1

            or epoch
            % VAL_EVERY
            == 0

            or epoch
            == EPOCHS
        )


        val_mse = None
        val_rmse = None
        val_mae = None


        if do_validation:

            (
                val_mse,
                val_rmse,
                val_mae
            ) = evaluate(
                model,
                val_loader
            )


            if val_mse < best_val:

                best_val = val_mse
                best_epoch = epoch

                evals_without_improvement = 0

                torch.save(
                    model.state_dict(),
                    checkpoint
                )

            else:

                evals_without_improvement += 1


        history.append(
            {
                "epoch":
                    epoch,

                "train_rmse":
                    float(
                        train_rmse
                    ),

                "val_rmse":
                    None
                    if val_rmse
                    is None
                    else float(
                        val_rmse
                    ),

                "val_mae":
                    None
                    if val_mae
                    is None
                    else float(
                        val_mae
                    )
            }
        )


        if do_validation:

            print(

                f"Epoch {epoch:03d} | "

                f"train RMSE "
                f"{train_rmse:.6e} Å | "

                f"val RMSE "
                f"{val_rmse:.6e} Å | "

                f"val MAE "
                f"{val_mae:.6e} Å | "

                f"{time.time()-epoch_start:.1f}s"
            )


        if (
            do_validation
            and
            evals_without_improvement
            >= PATIENCE_EVALS
        ):

            print(
                f"Early stopping at "
                f"epoch {epoch}. "
                f"Best epoch: "
                f"{best_epoch}"
            )

            break


    # ========================================================
    # Reload exact best checkpoint
    # ========================================================

    model.load_state_dict(

        torch.load(

            checkpoint,

            map_location=DEVICE,

            weights_only=True
        )
    )


    (
        val_mse,
        val_rmse,
        val_mae
    ) = evaluate(
        model,
        val_loader
    )


    elapsed = (
        time.time()
        - variant_start
    )


    print()
    print(
        f"BEST E3 VARIANT {variant}"
    )

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        f"Validation RMSE: "
        f"{val_rmse:.8f} Å"
    )

    print(
        f"Validation MAE:  "
        f"{val_mae:.8f} Å"
    )

    print(
        f"Wall time: "
        f"{elapsed/60:.2f} min"
    )


    with open(

        os.path.join(
            OUTPUT_DIR,
            f"history_e3_variant_{variant}.json"
        ),

        "w"

    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    del model
    del optimizer

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return {

        "variant":
            variant,

        "best_epoch":
            best_epoch,

        "val_mse":
            float(
                val_mse
            ),

        "val_rmse":
            float(
                val_rmse
            ),

        "val_mae":
            float(
                val_mae
            ),

        "parameters":
            parameter_count,

        "wall_seconds":
            float(
                elapsed
            ),

        "checkpoint":
            checkpoint
    }


# ============================================================
# Train E3 A/B/C
# ============================================================

experiment_start = time.time()

results = []

for variant in [
    "A",
    "B",
    "C"
]:

    results.append(
        train_variant(
            variant
        )
    )


# ============================================================
# A/B/C validation comparison
# ============================================================

print()
print(
    "=" * 72
)

print(
    "E3 A/B/C VALIDATION COMPARISON"
)

print(
    "=" * 72
)

print()

print(
    f"Zero baseline | "
    f"RMSE {zero_rmse:.8f} Å | "
    f"MAE {zero_mae:.8f} Å"
)


for r in results:

    improvement = (
        1.0
        - r["val_rmse"]
        / zero_rmse
    ) * 100.0

    print(

        f"{r['variant']} | "

        f"epoch "
        f"{r['best_epoch']:3d} | "

        f"RMSE "
        f"{r['val_rmse']:.8f} Å | "

        f"MAE "
        f"{r['val_mae']:.8f} Å | "

        f"vs-zero "
        f"{improvement:+.2f}%"
    )


# ============================================================
# Random SO(3) rotations
# ============================================================

def random_rotation_matrix(
    rng
):

    A = rng.normal(
        size=(
            3,
            3
        )
    )

    Q, R = np.linalg.qr(
        A
    )

    signs = np.sign(
        np.diag(
            R
        )
    )

    signs[
        signs == 0
    ] = 1.0

    Q = (
        Q
        @ np.diag(
            signs
        )
    )

    if np.linalg.det(
        Q
    ) < 0:

        Q[:, 0] *= -1.0

    return Q.astype(
        np.float32
    )


rng = np.random.default_rng(
    ROTATION_SEED
)

rotations = [

    random_rotation_matrix(
        rng
    )

    for _ in range(
        N_ROTATIONS
    )
]


print()
print(
    f"Generated "
    f"{N_ROTATIONS} "
    f"SO(3) rotations."
)


# ============================================================
# Rotated dataset
#
# IMPORTANT:
#
# val_cache currently contains normalized velocity/force.
#
# Because normalization is isotropic:
#
#       v_norm' = v_norm @ R.T
#
# except for the tiny empirical mean subtraction.
#
# For a strict rotation test, reconstruct physical vectors,
# rotate them, then normalize using the correspondingly
# rotated training mean.
#
# This ensures the preprocessing itself is equivariant.
# ============================================================

class RotatedEquivariantDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        cache,
        variant,
        R
    ):

        self.cache = cache
        self.variant = variant

        self.R = torch.tensor(
            R,
            dtype=torch.float32
        )


    def __len__(
        self
    ):

        return len(
            self.cache
        )


    def __getitem__(
        self,
        idx
    ):

        g = self.cache[idx]

        RT = self.R.T

        # Since g.velocity and g.force are already
        # isotropically normalized vectors, rotating
        # them directly is mathematically equivariant.

        velocity = (
            g.velocity
            @ RT
        )

        force = (
            g.force
            @ RT
        )

        edge_vector = (
            g.edge_vector
            @ RT
        )

        target = (
            g.y
            @ RT
        )


        return Data(
            species=g.species,
            velocity=velocity,
            force=force,
            edge_index=g.edge_index,
            edge_vector=edge_vector,
            edge_distance=g.edge_distance,
            y=target,
            num_nodes=g.species.shape[0]
        )

# ============================================================
# Prediction helper
# ============================================================

@torch.no_grad()
def predict_dataset(
    model,
    dataset
):

    loader = DataLoader(

        dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    predictions = []
    targets = []


    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        pred = model(
            batch
        )

        predictions.append(
            pred.float().cpu()
        )

        targets.append(
            batch.y.float().cpu()
        )


    return (
        torch.cat(
            predictions,
            dim=0
        ),

        torch.cat(
            targets,
            dim=0
        )
    )


def metric_dict(
    pred,
    target
):

    error = (
        pred
        - target
    )

    mse = torch.mean(
        error ** 2
    ).item()

    rmse = math.sqrt(
        mse
    )

    mae = torch.mean(
        torch.abs(
            error
        )
    ).item()


    return {
        "rmse":
            float(
                rmse
            ),

        "mae":
            float(
                mae
            )
    }


# ============================================================
# Rotation test for one variant
# ============================================================

def rotation_test_variant(
    result
):

    variant = result[
        "variant"
    ]

    print()
    print(
        "=" * 72
    )

    print(
        f"E3 ROTATION TEST — "
        f"VARIANT {variant}"
    )

    print(
        "=" * 72
    )


    model = E3DynamicsGNN(
        variant
    ).to(
        DEVICE
    )


    model.load_state_dict(

        torch.load(

            result[
                "checkpoint"
            ],

            map_location=DEVICE,

            weights_only=True
        )
    )


    model.eval()


    original_dataset = (
        EquivariantDataset(
            val_cache,
            variant
        )
    )


    (
        original_pred,
        original_target
    ) = predict_dataset(
        model,
        original_dataset
    )


    original_metrics = metric_dict(
        original_pred,
        original_target
    )


    print(
        f"Original RMSE: "
        f"{original_metrics['rmse']:.8f} Å"
    )


    # --------------------------------------------
    # Ensure checkpoint reproduction
    # --------------------------------------------

    difference = abs(
        original_metrics[
            "rmse"
        ]
        - result[
            "val_rmse"
        ]
    )


    print(
        f"Checkpoint difference: "
        f"{difference:.3e} Å"
    )


    if difference > 5e-6:

        raise RuntimeError(
            f"Variant {variant} checkpoint "
            f"failed reproduction gate."
        )


    rotation_results = []


    for rotation_id, R in enumerate(
        rotations,
        start=1
    ):

        rotated_dataset = (
            RotatedEquivariantDataset(
                val_cache,
                variant,
                R
            )
        )


        (
            rotated_pred,
            rotated_target
        ) = predict_dataset(
            model,
            rotated_dataset
        )


        accuracy = metric_dict(
            rotated_pred,
            rotated_target
        )


        R_tensor = torch.tensor(
            R,
            dtype=torch.float32
        )


        expected_rotated_pred = (
            original_pred
            @ R_tensor.T
        )


        equivariance = metric_dict(
            rotated_pred,
            expected_rotated_pred
        )


        degradation = (
            accuracy[
                "rmse"
            ]
            / original_metrics[
                "rmse"
            ]
        )


        rotation_results.append(
            {
                "rotation_id":
                    rotation_id,

                "rotation_matrix":
                    R.tolist(),

                "determinant":
                    float(
                        np.linalg.det(
                            R
                        )
                    ),

                "rotated_target_rmse":
                    accuracy[
                        "rmse"
                    ],

                "rotated_target_mae":
                    accuracy[
                        "mae"
                    ],

                "equivariance_rmse":
                    equivariance[
                        "rmse"
                    ],

                "equivariance_mae":
                    equivariance[
                        "mae"
                    ],

                "accuracy_degradation_factor":
                    float(
                        degradation
                    )
            }
        )


        print(

            f"Rotation "
            f"{rotation_id:02d} | "

            f"target RMSE "
            f"{accuracy['rmse']:.8f} Å | "

            f"equiv RMSE "
            f"{equivariance['rmse']:.8e} Å | "

            f"degradation "
            f"{degradation:.4f}x"
        )


    rotated_rmse = np.array(
        [
            r[
                "rotated_target_rmse"
            ]
            for r in rotation_results
        ]
    )


    equiv_rmse = np.array(
        [
            r[
                "equivariance_rmse"
            ]
            for r in rotation_results
        ]
    )


    degradation = (
        rotated_rmse
        / original_metrics[
            "rmse"
        ]
    )


    summary = {

        "variant":
            variant,

        "original_rmse":
            original_metrics[
                "rmse"
            ],

        "rotated_rmse_mean":
            float(
                rotated_rmse.mean()
            ),

        "rotated_rmse_std":
            float(
                rotated_rmse.std(
                    ddof=1
                )
            ),

        "equivariance_rmse_mean":
            float(
                equiv_rmse.mean()
            ),

        "equivariance_rmse_std":
            float(
                equiv_rmse.std(
                    ddof=1
                )
            ),

        "degradation_mean":
            float(
                degradation.mean()
            ),

        "degradation_min":
            float(
                degradation.min()
            ),

        "degradation_max":
            float(
                degradation.max()
            ),

        "rotations":
            rotation_results
    }


    print()
    print(
        f"VARIANT {variant} "
        f"ROTATION SUMMARY"
    )

    print(
        f"Original RMSE: "
        f"{original_metrics['rmse']:.8f} Å"
    )

    print(
        f"Rotated RMSE:  "
        f"{rotated_rmse.mean():.8f} "
        f"± "
        f"{rotated_rmse.std(ddof=1):.8f} Å"
    )

    print(
        f"Equiv RMSE:    "
        f"{equiv_rmse.mean():.8e} "
        f"± "
        f"{equiv_rmse.std(ddof=1):.8e} Å"
    )

    print(
        f"Degradation:   "
        f"{degradation.mean():.5f}x"
    )


    del model

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return summary


# ============================================================
# Run rotation tests for A/B/C
# ============================================================

rotation_summaries = []

for result in results:

    rotation_summaries.append(
        rotation_test_variant(
            result
        )
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print(
    "=" * 90
)

print(
    "FINAL E3NN A/B/C + ROTATION COMPARISON"
)

print(
    "=" * 90
)


print()
print(
    f"Zero predictor RMSE: "
    f"{zero_rmse:.8f} Å"
)

print()


for result, rotation in zip(
    results,
    rotation_summaries
):

    improvement = (
        1.0
        - result[
            "val_rmse"
        ]
        / zero_rmse
    ) * 100.0


    print(

        f"{result['variant']} | "

        f"val RMSE "
        f"{result['val_rmse']:.8f} Å | "

        f"vs-zero "
        f"{improvement:+.2f}% | "

        f"rot RMSE "
        f"{rotation['rotated_rmse_mean']:.8f} Å | "

        f"equiv RMSE "
        f"{rotation['equivariance_rmse_mean']:.3e} Å | "

        f"rot degradation "
        f"{rotation['degradation_mean']:.5f}x"
    )


# ============================================================
# Save complete experiment
# ============================================================

total_minutes = (
    time.time()
    - experiment_start
) / 60.0


output = {

    "experiment":
        "e3nn_abc_plus_rotation",

    "torch_version":
        torch.__version__,

    "e3nn_version":
        e3nn.__version__,

    "seed":
        SEED,

    "rotation_seed":
        ROTATION_SEED,

    "n_rotations":
        N_ROTATIONS,

    "train_samples":
        len(train_paths),

    "val_samples":
        len(val_paths),

    "test_set_used":
        False,

    "zero_baseline_rmse":
        float(
            zero_rmse
        ),

    "zero_baseline_mae":
        float(
            zero_mae
        ),

    "hidden_irreps":
        str(
            HIDDEN_IRREPS
        ),

    "edge_irreps":
        str(
            EDGE_IRREPS
        ),

    "output_irreps":
        str(
            OUTPUT_IRREPS
        ),

    "results":
        results,

    "rotation_results":
        rotation_summaries,

    "total_minutes":
        float(
            total_minutes
        )
}


OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "e3nn_abc_rotation_results.json"
)


with open(
    OUTPUT_FILE,
    "w"
) as f:

    json.dump(
        output,
        f,
        indent=2
    )


print()
print(
    f"Total experiment time: "
    f"{total_minutes:.2f} min"
)

print(
    "Saved:",
    OUTPUT_FILE
)

print()

print(
    "TEST SET HAS NOT BEEN "
    "LOADED OR EVALUATED."
)